# LoRA-RoBERTa Token Classification Pipeline (NER Novel Frankenstein)

Notebook ini disusun sebagai **panduan interaktif dan panduan belajar** untuk mempelajari seluruh alur kerja penelitian *Named Entity Recognition* (NER) pada novel klasik *Frankenstein* karya Mary Shelley menggunakan **Parameter-Efficient Fine-Tuning (PEFT) dengan algoritma LoRA (Low-Rank Adaptation)** pada arsitektur `roberta-base`.

---

## 🔄 Alur Eksperimen (Sesuai README.md)

1. **FASE A: Penyiapan Korpus & Koreksi Data**
   - Langkah 1: Scraping Teks Novel Frankenstein (`src/scrape_gutenberg.py`)
   - Langkah 2: Pelabelan Awal Otomatis OntoNotes 5.0 (`src/annotate_data.py`)
   - Langkah 3: Koreksi Bias Label Dataset untuk Hidden Entities (`scratch/patch_dataset.py`)

2. **FASE B: Setup Model & Pelatihan Komparatif (3 Metode)**
   - Langkah 4: Preprocessing Data Loader & Token Alignment BPE (`src/data_loader.py`)
   - Langkah 5: Injeksi PEFT LoRA & Setup Model RoBERTa Baseline (`src/model.py`)
   - Langkah 6: Prosedur Pelatihan & Grid Search Sweep (`src/train.py` & `src/main.py`)
     - *Metode 1:* Baseline Model (Full Fine-Tuning / Tanpa LoRA)
     - *Metode 2:* Regular LoRA Model ($r=8, \alpha=16$)
     - *Metode 3:* Hyperparameter Tuning Grid Search Sweep (12 Kombinasi)

3. **FASE C: Evaluasi Multi-Dimensi & Visualisasi Hasil**
   - Langkah 7: Evaluasi Kebahasaan, MUC-5, Fine-Grained, & Hidden Entity Recall (`src/evaluate.py`)
   - Langkah 8: Whole Dataset Inference & Prediksi Kalimat (`src/predict.py`)
   - Langkah 9: Visualisasi 6 Grafik Plot Analitis (`src/visualize_results.py`)
   - Langkah 10: Rekonstruksi & Konsolidasi Log Pelatihan (`scratch/reconstruct_logs.py`)

## 🛠️ Instalasi, Import Dependensi & Resolusi Path Otomatis

Memuat seluruh modul Python yang dibutuhkan dan mengonfigurasi direktori kerja (`PROJECT_ROOT`, `DATA_DIR`, `RESULTS_DIR`) secara otomatis agar bebas dari error `FileNotFoundError` baik di VS Code, Jupyter Notebook, maupun Google Colab.

In [ ]:
import os
import sys
import json
import time
import re
import shutil
import urllib.request
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    RobertaTokenizerFast, 
    RobertaForTokenClassification, 
    AutoTokenizer,
    Trainer, 
    TrainingArguments, 
    DataCollatorForTokenClassification
)
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.metrics.sequence_labeling import get_entities

# Set environment
os.environ["TRANSFORMERS_NO_TF"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Automatic Project Root Resolution
CURRENT_DIR = os.getcwd()
if os.path.basename(CURRENT_DIR) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
else:
    PROJECT_ROOT = os.path.abspath(CURRENT_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"[Device] Running on: '{device}'")
print(f"[Project Root] {PROJECT_ROOT}")
print(f"[Data Dir]     {DATA_DIR}")
print(f"[Results Dir]  {RESULTS_DIR}")

---
## 📥 FASE A: Penyiapan Korpus & Koreksi Data

### Langkah 1: Scraping Teks Novel Frankenstein dari Gutenberg
Mengunduh teks novel *Frankenstein* versi HTML dari Project Gutenberg, membersihkan tag HTML, dan memilah isi teks per bab. Jika berkas data mentah sudah ada, tahap ini akan dilewati secara otomatis.

In [ ]:
def scrape_frankenstein(output_json_path=None):
    if output_json_path is None:
        output_json_path = os.path.join(DATA_DIR, "frankenstein_data.json")
        
    if os.path.exists(output_json_path):
        print(f"Raw data file already present at '{output_json_path}'. Skipping web scraping.")
        return
        
    url = "https://www.gutenberg.org/files/84/84-h/84-h.htm"
    print(f"Fetching novel from Project Gutenberg: {url}")
    
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        html_doc = urllib.request.urlopen(req).read().decode('utf-8')
    except Exception as e:
        print(f"Failed to fetch from Gutenberg web: {e}")
        return
        
    soup = BeautifulSoup(html_doc, 'html.parser')
    chapters_data = []
    current_chapter = "Preface/Letter"
    current_text = []
    
    for element in soup.find_all(['h2', 'h3', 'p']):
        text = element.get_text().strip()
        if not text:
            continue
            
        if element.name in ['h2', 'h3'] and ('chapter' in text.lower() or 'letter' in text.lower()):
            if current_text:
                chapters_data.append({
                    "chapter": current_chapter,
                    "text": " ".join(current_text)
                })
                current_text = []
            current_chapter = text
        else:
            current_text.append(text)
            
    if current_text:
        chapters_data.append({
            "chapter": current_chapter,
            "text": " ".join(current_text)
        })
        
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(chapters_data, f, indent=4)
        
    print(f"Successfully scraped {len(chapters_data)} chapters to '{output_json_path}'")

# Run scraping
scrape_frankenstein()

### Langkah 2: Pelabelan Awal Otomatis (BERT OntoNotes 5.0)
Menggunakan model besar `tner/bert-base-ontonotes5` untuk melabeli kata-kata pada teks mentah ke dalam format skema BIO. Jika berkas anotasi sudah ada di folder `data/`, tahap ini otomatis dilewati.

In [ ]:
def auto_annotate_corpus(raw_json=None, output_annotated=None):
    if raw_json is None:
        raw_json = os.path.join(DATA_DIR, "frankenstein_data.json")
    if output_annotated is None:
        output_annotated = os.path.join(DATA_DIR, "frankenstein_annotated_auto.json")
        
    final_annotated = os.path.join(DATA_DIR, "frankenstein_annotated.json")
    if os.path.exists(output_annotated) or os.path.exists(final_annotated):
        print("Annotated dataset already present in Data Directory. Skipping auto-annotation.")
        return
        
    if not os.path.exists(raw_json):
        print(f"File raw data '{raw_json}' not found. Please run Step 1 first.")
        return
        
    print(f"Loading raw data from '{raw_json}'...")
    with open(raw_json, "r", encoding="utf-8") as f:
        chapters = json.load(f)
        
    print("Loading pre-trained NER model ('tner/bert-base-ontonotes5')...")
    from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
    tokenizer = AutoTokenizer.from_pretrained("tner/bert-base-ontonotes5")
    model = AutoModelForTokenClassification.from_pretrained("tner/bert-base-ontonotes5")
    ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)
    
    annotated_dataset = []
    for ch in chapters:
        text = ch.get("text", "")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        for sent in sentences:
            sent = sent.strip()
            if len(sent) < 10:
                continue
            entities = ner_pipeline(sent)
            words = sent.split()
            tags = ["O"] * len(words)
            for ent in entities:
                ent_text = ent['word'].strip()
                ent_label = ent['entity_group']
                for idx, w in enumerate(words):
                    clean_w = w.strip(PUNCT_CHARS)
                    if ent_text.lower() in clean_w.lower() and clean_w != "":
                        tags[idx] = f"B-{ent_label}"
                        break
            annotated_dataset.append({"tokens": words, "tags": tags})
            
    with open(output_annotated, "w", encoding="utf-8") as f:
        json.dump(annotated_dataset, f, indent=4)
    print(f"Successfully generated {len(annotated_dataset)} annotated sentences in '{output_annotated}'")

# Run auto annotation
auto_annotate_corpus()

### Langkah 3: Koreksi Bias Label Dataset (Krusial untuk Hidden Entities)
Mengoreksi bias label pada dataset mentah. Sebanyak **154 kata benda sastra penunjuk tokoh** (*monster, creature, wretch, fiend, demon, creator*) yang tadinya salah terlabeli sebagai `O` dikoreksi secara eksplisit menjadi `B-PERSON` / `I-PERSON`.

In [ ]:
def patch_dataset_bias(input_file=None, output_file=None):
    if input_file is None:
        input_file = os.path.join(DATA_DIR, "frankenstein_annotated_auto.json")
    if output_file is None:
        output_file = os.path.join(DATA_DIR, "frankenstein_annotated.json")
        
    target_keywords = ["monster", "creature", "wretch", "fiend", "demon", "creator"]
    
    if os.path.exists(output_file):
        print(f"Patched dataset already present at '{output_file}'. Ready for training!")
        return
        
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        return
        
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    patched_count = 0
    for sample in data:
        tokens = sample.get("tokens", [])
        tags = sample.get("tags", [])
        for idx, word in enumerate(tokens):
            clean_word = word.lower().strip(PUNCT_CHARS)
            if clean_word in target_keywords:
                prev_is_person = False
                if idx > 0 and tags[idx - 1] in ["B-PERSON", "I-PERSON"]:
                    prev_is_person = True
                new_tag = "I-PERSON" if prev_is_person else "B-PERSON"
                if tags[idx] != new_tag:
                    tags[idx] = new_tag
                    patched_count += 1
                    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)
    print(f"Successfully patched {patched_count} hidden entity tokens in '{output_file}'")

# Run patch dataset
patch_dataset_bias()

---
## 📦 FASE B: Setup Model & Pelatihan Komparatif

### Langkah 4: Data Loader & Alignment Subwords (RoBERTa Byte-Pair Encoding)
Membangun `NERDataset` PyTorch, memuat tag mapping, dan menyelaraskan label BIO dengan subwords RoBERTa menggunakan label `-100`.

In [ ]:
class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Successfully loaded {len(data)} examples from '{file_path}'")
    return data

def get_tag_mappings(data):
    unique_tags = set()
    for item in data:
        unique_tags.update(item["tags"])
        
    sorted_tags = sorted(list(unique_tags))
    if "O" in sorted_tags:
        sorted_tags.remove("O")
        sorted_tags = ["O"] + sorted_tags
        
    tag2id = {tag: i for i, tag in enumerate(sorted_tags)}
    id2tag = {i: tag for i, tag in enumerate(sorted_tags)}
    return tag2id, id2tag

def preprocess_and_tokenize(data, tokenizer: RobertaTokenizerFast, tag2id):
    texts = [x["tokens"] for x in data]
    tags = [x["tags"] for x in data]
    
    tokenized_inputs = tokenizer(
        texts,
        is_split_into_words=True,
        truncation=True,
        padding=True
    )
    
    labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(tag2id.get(label[word_idx], tag2id.get("O", 0)))
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Load data and prepare mappings
dataset_path = os.path.join(DATA_DIR, "frankenstein_annotated.json")
raw_data = load_data(dataset_path)
tag2id, id2tag = get_tag_mappings(raw_data)
print(f"Total Unique Classes: {len(tag2id)}")
print(f"Tag Mappings: {tag2id}")

# Save tag mappings to results dir for predict script
mappings_path = os.path.join(RESULTS_DIR, "tag_mappings.json")
with open(mappings_path, "w", encoding="utf-8") as f:
    json.dump({"tag2id": tag2id, "id2tag": {str(k): v for k, v in id2tag.items()}}, f, indent=4)
print(f"Saved tag mappings to '{mappings_path}'")

### Langkah 5: Injeksi PEFT LoRA & Model Setup
Fungsi pembangun model RoBERTa dengan injeksi adaptor LoRA (hanya melatih parameter Query & Value) serta model Baseline Full Fine-Tuning.

In [ ]:
def get_roberta_lora_model(model_name="roberta-base", num_labels=19, r=8, lora_alpha=16, lora_dropout=0.1):
    """
    Load pre-trained RoBERTa model for token classification, freeze base weights,
    and inject LoRA parameters into Query & Value modules.
    """
    model = RobertaForTokenClassification.from_pretrained(model_name, num_labels=num_labels)
    peft_config = LoraConfig(
        task_type=TaskType.TOKEN_CLS,
        inference_mode=False,
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=["query", "value"]
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    return model

def get_roberta_baseline_model(model_name="roberta-base", num_labels=19):
    """
    Load standard pre-trained RoBERTa model for full fine-tuning (without LoRA).
    """
    model = RobertaForTokenClassification.from_pretrained(model_name, num_labels=num_labels)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_param = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable_params:,} || all params: {all_param:,} || trainable%: {100 * trainable_params / all_param:.4f}%")
    return model

# Test model instantiations
print("--- LoRA Model Setup ---")
test_lora_model = get_roberta_lora_model(num_labels=len(tag2id), r=8, lora_alpha=64)

print("\n--- Baseline Full Fine-Tuning Setup ---")
test_baseline_model = get_roberta_baseline_model(num_labels=len(tag2id))

### Langkah 6: Modul Evaluator Kebahasaan, MUC-5, Fine-Grained, & Hidden Entities
Mengkompilasi seluruh fungsi ukur evaluasi yang digunakan dalam penelitian.

In [ ]:
def evaluate_linguistic_metrics(predictions, references):
    report = classification_report(references, predictions)
    f1 = f1_score(references, predictions)
    return report, f1

def evaluate_muc5_errors(predictions, references):
    cor, inc, mis, spu = 0, 0, 0, 0
    for pred_seq, ref_seq in zip(predictions, references):
        ref_ents = get_entities(ref_seq)
        pred_ents = get_entities(pred_seq)
        ref_by_boundary = {(start, end): ent_type for ent_type, start, end in ref_ents}
        pred_by_boundary = {(start, end): ent_type for ent_type, start, end in pred_ents}
        for boundary, ref_type in ref_by_boundary.items():
            if boundary in pred_by_boundary:
                pred_type = pred_by_boundary[boundary]
                if ref_type == pred_type:
                    cor += 1
                else:
                    inc += 1
            else:
                mis += 1
        for boundary in pred_by_boundary:
            if boundary not in ref_by_boundary:
                spu += 1
    return {"COR": cor, "INC": inc, "MIS": mis, "SPU": spu}

def evaluate_fine_grained(predictions, references, sentences=None):
    from collections import Counter
    ref_entities_list = []
    entity_text_labels = {}
    entity_text_freq = Counter()
    
    for sent_idx, ref_seq in enumerate(references):
        ref_ents = get_entities(ref_seq)
        for ent_type, start, end in ref_ents:
            if sentences and sent_idx < len(sentences):
                text = " ".join(sentences[sent_idx][start:end]).lower()
            else:
                text = f"entity_{start}_{end}"
            ref_entities_list.append((sent_idx, start, end, ent_type, text))
            entity_text_freq[text] += 1
            if text not in entity_text_labels:
                entity_text_labels[text] = []
            entity_text_labels[text].append(ent_type)
            
    entity_consistency = {}
    for text, labels in entity_text_labels.items():
        most_common_count = Counter(labels).most_common(1)[0][1]
        entity_consistency[text] = most_common_count / len(labels)
        
    pred_entities_by_sent = []
    for pred_seq in predictions:
        pred_ents = get_entities(pred_seq)
        pred_entities_by_sent.append({(start, end): ent_type for ent_type, start, end in pred_ents})
        
    categories = {
        "eLen_short": {"total": 0, "correct": 0},
        "eLen_long": {"total": 0, "correct": 0},
        "eCon_consistent": {"total": 0, "correct": 0},
        "eCon_inconsistent": {"total": 0, "correct": 0},
        "eFre_few_shot": {"total": 0, "correct": 0},
        "eFre_many_shot": {"total": 0, "correct": 0}
    }
    
    for sent_idx, start, end, ref_type, text in ref_entities_list:
        correct = False
        if sent_idx < len(pred_entities_by_sent):
            pred_sent = pred_entities_by_sent[sent_idx]
            if (start, end) in pred_sent and pred_sent[(start, end)] == ref_type:
                correct = True
        length = end - start
        len_cat = "eLen_long" if length >= 4 else "eLen_short"
        categories[len_cat]["total"] += 1
        if correct:
            categories[len_cat]["correct"] += 1
        con = entity_consistency[text]
        con_cat = "eCon_consistent" if con == 1.0 else "eCon_inconsistent"
        categories[con_cat]["total"] += 1
        if correct:
            categories[con_cat]["correct"] += 1
        freq = entity_text_freq[text]
        fre_cat = "eFre_few_shot" if freq <= 2 else "eFre_many_shot"
        categories[fre_cat]["total"] += 1
        if correct:
            categories[fre_cat]["correct"] += 1
            
    results = {}
    for cat, stats in categories.items():
        total = stats["total"]
        correct = stats["correct"]
        acc = correct / total if total > 0 else 0.0
        results[cat] = {"total": total, "correct": correct, "accuracy": acc}
    return results

def evaluate_hidden_entities(predictions, references, sentences):
    hidden_keywords = ["monster", "creature", "wretch", "fiend", "demon", "creator"]
    stats = {k: {"total": 0, "detected": 0} for k in hidden_keywords}
    
    for pred_seq, ref_seq, words in zip(predictions, references, sentences):
        for w, true_t, pred_t in zip(words, ref_seq, pred_seq):
            clean_word = w.lower().strip(PUNCT_CHARS)
            if clean_word in hidden_keywords:
                stats[clean_word]["total"] += 1
                if "PERSON" in pred_t:
                    stats[clean_word]["detected"] += 1
                    
    total_all = sum(stats[k]["total"] for k in hidden_keywords)
    detected_all = sum(stats[k]["detected"] for k in hidden_keywords)
    overall_recall = (detected_all / total_all * 100.0) if total_all > 0 else 0.0
    detailed_recall = {k: (stats[k]["detected"] / stats[k]["total"] * 100.0) if stats[k]["total"] > 0 else 0.0 for k in hidden_keywords}
    
    return {
        "overall_recall": overall_recall,
        "total_hidden": total_all,
        "detected_hidden": detected_all,
        "detailed_recall": detailed_recall
    }

### Langkah 7: Panduan Eksekusi Pelatihan 3 Metode Komparatif

Untuk menjalankan pelatihan pada dataset penuh melalui CLI/terminal:

```cmd
# 1. Metode Baseline Full Fine-Tuning (Tanpa LoRA)
python src/main.py --data_path data/frankenstein_annotated.json --output_dir ./results --no_lora --epochs 3 --batch_size 8

# 2. Metode Regular LoRA (r=8, alpha=16)
python src/main.py --data_path data/frankenstein_annotated.json --output_dir ./results --epochs 3 --batch_size 8

# 3. Metode LoRA Grid Search Sweep (12 Kombinasi)
python src/main.py --data_path data/frankenstein_annotated.json --output_dir ./results --grid_search
```

---
## 📊 FASE C: Evaluasi Multi-Dimensi & Visualisasi Grafik

### Langkah 8: Inferensi & Prediksi Teks Novel Utuh
Script inferensi cerdas yang secara dinamis memuat model PEFT/LoRA maupun model Full Fine-Tuning standar berdasarkan ketersediaan berkas `adapter_config.json`.

In [ ]:
def predict_entities(text, model_dir=None, mappings_path=None):
    if model_dir is None:
        best_model_path = os.path.join(RESULTS_DIR, "best_model")
        std_model_path = os.path.join(RESULTS_DIR, "standard_run")
        model_dir = best_model_path if os.path.exists(best_model_path) else std_model_path
        
    if mappings_path is None:
        mappings_path = os.path.join(RESULTS_DIR, "tag_mappings.json")
        
    if not os.path.exists(model_dir):
        print(f"Model directory not found at '{model_dir}'. Please run training first.")
        return
        
    local_id2tag = {}
    if os.path.exists(mappings_path):
        with open(mappings_path, "r", encoding="utf-8") as f:
            mappings = json.load(f)
            local_id2tag = {int(k): v for k, v in mappings.get("id2tag", {}).items()}
    elif "id2tag" in globals():
        local_id2tag = globals()["id2tag"]
    else:
        local_id2tag = {0: "O", 1: "B-PERSON", 2: "I-PERSON"}
        
    num_labels = len(local_id2tag) if local_id2tag else 19
    tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base", add_prefix_space=True)
    adapter_config_path = os.path.join(model_dir, "adapter_config.json")
    
    if os.path.exists(adapter_config_path):
        print(f"[Loading Mode] PEFT/LoRA adapter detected in '{model_dir}'")
        base_model = RobertaForTokenClassification.from_pretrained("roberta-base", num_labels=num_labels)
        model = PeftModel.from_pretrained(base_model, model_dir)
    else:
        print(f"[Loading Mode] Standard Full Fine-Tuning weights detected in '{model_dir}'")
        model = RobertaForTokenClassification.from_pretrained(model_dir, num_labels=num_labels)
        
    model.to(device)
    model.eval()
    
    words = text.split()
    inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)[0].tolist()
        
    word_ids = inputs.word_ids(batch_index=0)
    previous_word_idx = None
    predicted_tags = []
    
    for idx, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx != previous_word_idx:
            tag_id = predictions[idx]
            predicted_tags.append(local_id2tag.get(tag_id, "O"))
            previous_word_idx = word_idx
            
    print(f"\n--- Prediction Result using '{os.path.basename(model_dir)}' ---")
    for w, t in zip(words, predicted_tags):
        print(f"{w:<15} -> {t}")

# Test sample sentence prediction
sample_text = "I saw Victor Frankenstein and the monster in Geneva."
predict_entities(sample_text)

### Langkah 9: Visualisasi 6 Grafik Plot Analitis
Membaca berkas ringkasan CSV untuk menghasilkan 6 grafik plot analitis (Heatmap F1-score, Galat MUC-5, Ketahanan Fine-Grained, Garis Waktu Pelatihan, Perbandingan F1 vs Hidden Recall, dan Efisiensi Waktu vs VRAM).

In [ ]:
def generate_visualizations(grid_csv_path=None, output_dir=None):
    if grid_csv_path is None:
        grid_csv_path = os.path.join(RESULTS_DIR, "grid_search_results.csv")
    if output_dir is None:
        output_dir = os.path.join(RESULTS_DIR, "plots")
        
    if not os.path.exists(grid_csv_path):
        print(f"CSV file not found at '{grid_csv_path}'. Run grid search training first.")
        return
        
    df = pd.read_csv(grid_csv_path)
    os.makedirs(output_dir, exist_ok=True)
    sns.set_theme(style="whitegrid")
    plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 14})
    
    # Plot 1: Heatmap
    pivot_f1 = df.pivot(index="rank", columns="alpha", values="f1")
    plt.figure(figsize=(8, 6))
    sns.heatmap(pivot_f1, annot=True, fmt=".4f", cmap="YlGnBu", cbar_kws={'label': 'Macro F1-Score'})
    plt.title("Pengaruh Kombinasi Rank (r) dan Alpha (α) terhadap F1-Score")
    plt.xlabel("Alpha (α)")
    plt.ylabel("Rank (r)")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "heatmap_rank_alpha_f1.png"), dpi=300)
    plt.close()
    
    # Plot 5 & 6: Method Comparisons
    no_lora_csv = os.path.join(RESULTS_DIR, "standard_no_lora_results.csv")
    lora_csv = os.path.join(RESULTS_DIR, "standard_lora_results.csv")
    
    if os.path.exists(no_lora_csv) and os.path.exists(lora_csv):
        df_no_lora = pd.read_csv(no_lora_csv)
        df_lora = pd.read_csv(lora_csv)
        best_row = df.loc[df["f1"].idxmax()]
        
        methods_data = [
            {"Metode": "Full FT (No LoRA)", "F1": df_no_lora.loc[0, "f1"]*100, "Hidden": df_no_lora.loc[0, "hidden_entity_recall"], "Time": df_no_lora.loc[0, "training_time_sec"], "VRAM": df_no_lora.loc[0, "peak_vram_gb"]},
            {"Metode": "Regular LoRA (r=8, a=16)", "F1": df_lora.loc[0, "f1"]*100, "Hidden": df_lora.loc[0, "hidden_entity_recall"], "Time": df_lora.loc[0, "training_time_sec"], "VRAM": df_lora.loc[0, "peak_vram_gb"]},
            {"Metode": f"Best LoRA (r={int(best_row['rank'])}, a={int(best_row['alpha'])})", "F1": best_row["f1"]*100, "Hidden": best_row["hidden_entity_recall"], "Time": best_row["training_time_sec"], "VRAM": best_row["peak_vram_gb"]}
        ]
        df_m = pd.DataFrame(methods_data)
        
        # Plot Bar Comparison
        plt.figure(figsize=(10, 6))
        x = np.arange(len(df_m["Metode"]))
        width = 0.35
        plt.bar(x - width/2, df_m["F1"], width, label="Overall F1-Score (%)", color="#9b59b6")
        plt.bar(x + width/2, df_m["Hidden"], width, label="Hidden Entity Recall (%)", color="#e74c3c")
        plt.title("Perbandingan F1-Score dan Hidden Entity Recall antar Metode")
        plt.xlabel("Metode Pelatihan")
        plt.ylabel("Performa (%)")
        plt.xticks(x, df_m["Metode"])
        plt.ylim(0, 110)
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, "method_comparison_f1_hidden.png"), dpi=300)
        plt.close()
        
    print(f"Generated analytical visualization plots in '{output_dir}'")

# Run visualization function
generate_visualizations()

### Langkah 10: Rekonstruksi & Konsolidasi Log Pelatihan
Menggabungkan riwayat log latih dari seluruh eksperimen (Baseline FFT, Regular LoRA, dan 12 Grid Search combination) ke dalam berkas `results/full_training_log.txt`.

In [ ]:
def reconstruct_logs(results_dir=None, output_file=None):
    if results_dir is None:
        results_dir = RESULTS_DIR
    if output_file is None:
        output_file = os.path.join(RESULTS_DIR, "full_training_log.txt")
        
    if not os.path.exists(results_dir):
        print(f"Results directory '{results_dir}' not found.")
        return
        
    standard_dirs = ["standard_run_no_lora", "standard_run"]
    lora_dirs = [d for d in os.listdir(results_dir) if d.startswith("lora_") and os.path.isdir(os.path.join(results_dir, d))]
    
    def parse_lora_name(name):
        match = re.match(r"lora_r(\d+)_a(\d+)", name)
        return (int(match.group(1)), int(match.group(2))) if match else (999, 999)
        
    lora_dirs.sort(key=parse_lora_name)
    all_dirs = []
    
    for sd in standard_dirs:
        if os.path.exists(os.path.join(results_dir, sd)):
            all_dirs.append((sd, sd.replace("_", " ").upper()))
    for ld in lora_dirs:
        r, a = parse_lora_name(ld)
        all_dirs.append((ld, f"LoRA Grid Search (rank={r}, alpha={a})"))
        
    with open(output_file, "w", encoding="utf-8") as out:
        out.write("========================================================================\n")
        out.write("RECONSTRUCTED COMPLETE TRAINING LOGS FROM ALL EXPERIMENTS\n")
        out.write("========================================================================\n\n")
        
        for dir_name, display_title in all_dirs:
            dir_path = os.path.join(results_dir, dir_name)
            checkpoints = [c for c in os.listdir(dir_path) if c.startswith("checkpoint-") and os.path.isdir(os.path.join(dir_path, c))]
            if not checkpoints:
                continue
            checkpoints.sort(key=lambda x: int(x.split("-")[1]))
            state_file = os.path.join(dir_path, checkpoints[-1], "trainer_state.json")
            if not os.path.exists(state_file):
                continue
            with open(state_file, "r") as f:
                state_data = json.load(f)
            out.write(f"========================================================================\n")
            out.write(f"Training Config: {display_title}\n")
            out.write(f"========================================================================\n\n")
            for log in state_data.get("log_history", []):
                if "loss" in log:
                    out.write(f"{{'loss': {log['loss']:.4f}, 'learning_rate': {log.get('learning_rate')}, 'epoch': {log.get('epoch'):.2f}}}\n")
            out.write("\n\n")
            
    print(f"Reconstructed training logs saved to: '{output_file}'")

# Run log reconstruction
reconstruct_logs()

---
## 🎯 Kesimpulan & Ringkasan Hasil Utama

Berdasarkan alur eksperimen komprehensif di atas, hasil penelitian skripsi merumuskan poin-poin kunci sebagai berikut:

1. **Efisiensi Komputasi LoRA:**  
   Metode Best LoRA ($r=8, \alpha=64$) memotong alokasi parameter aktif hingga **0,237%** (294 ribu parameter dibanding 124 juta parameter RoBERTa). Hal ini memangkas konsumsi **Peak VRAM GPU hingga 47,56%** (hanya **1,24 GB**) dan mempercepat **Waktu Pelatihan hingga 41,20%** (**620,47 detik**).

2. **Akurasi & Hidden Entity Recall:**  
   Best LoRA mencapai F1-Score **78,26%** (hanya terpaut 5,63% dari Full Fine-Tuning baseline) dan meraih **94,16% Hidden Entity Recall** secara murni dari representasi model tanpa intervensi aturan (*rule-based*).

3. **Pencegahan Catastrophic Forgetting:**  
   Dengan membekukan 99,76% bobot asli model dasar, LoRA mencegah degradasi pengetahuan umum bahasa Inggris yang dialami oleh Full Fine-Tuning.